# Mesh Segmentation

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import igl
import meshplot as mp

from cutting.hierarchical import mesh_segment
from utils.mesh_utils import write_ply_file, write_proxy_cluster, write_region_cluster
from patch_segment.core import pre_split

mp.jupyter()


## Configuration


In [ ]:
obj_path = "example_data/selected/00023435_385b221a0d58490b84f3edee_trimesh_000.obj"

INPUT_MESH = Path(obj_path)
OUTPUT_DIR = Path(f'example_results/{INPUT_MESH.stem}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VISUALIZE_VSA = True
VISUALIZE_PARTS = False


## 1. pre-segmentation


In [ ]:
v, f = igl.read_triangle_mesh(str(INPUT_MESH))
print(f"load mesh: {INPUT_MESH.stem}, #v: {v.shape[0]}, #f: {f.shape[0]}")

proxy_cluster, region_cluster, face_color = pre_split(
    v, f, visualize=VISUALIZE_VSA, visualize_part=VISUALIZE_PARTS
)

write_proxy_cluster(proxy_cluster, OUTPUT_DIR / "proxy_cluster.txt")
# write_region_cluster(region_cluster, OUTPUT_DIR / "region_cluster.txt")
write_ply_file(v, f, face_color, OUTPUT_DIR / "pre_segmented.ply")


## 2. Hierarchical patch graph construction and cutting


In [ ]:
ITERATION_TIME = 2

objects = mesh_segment(
    v, f, ITERATION_TIME, patch2f=proxy_cluster
)
print(f"segmented objects: {len(objects)}")
